In [95]:
!pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ast

# 경고 무시
warnings.filterwarnings("ignore")
%config lnlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
	from google.colab import drive
	drive.mount('/content/drive')

	import os
	os.chdir('/content/drive/MyDrive/파트4')
	print('✅ Succesful access google_drive_directory')
	
except Exception as e:
	print('❌')
	
	
## get_df 함수
def get_df(db_name, table_name):
    table_name = pd.read_csv(
        f"gs://high_project/{db_name}/{table_name}.csv",
        storage_options={'token' : API_KEY_PATH}
        )
    return table_name
    

## literal_eval 형변환 함수
import ast
def to_literal_eval(df, column):
    df[column] = df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])
    return df
    
drop_users = [831956, 1580627, 1580689, 1580626, 995177]
	

❌


In [4]:
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

def get_df(db_name, table_name):
    table_name = pd.read_csv(
        f"gs://high_project/{db_name}/{table_name}.csv",
        storage_options={'token' : API_KEY_PATH}
        )
    return table_name

In [69]:
# 데이터 불러오기
accounts_blockrecord = get_df('votes', 'accounts_blockrecord')

# 컬럼 순서 정리
accounts_blockrecord = accounts_blockrecord[['id', 'user_id', 'block_user_id', 'reason', 'created_at']]

# 시간타입 형변환
accounts_blockrecord['created_at'] = pd.to_datetime(accounts_blockrecord['created_at'])

# ‼️ 중복치 제거
accounts_blockrecord = accounts_blockrecord.loc[:, 'user_id':].drop_duplicates()

# ‼️ 관리자 유저 필터링
accounts_blockrecord = accounts_blockrecord[~accounts_blockrecord['user_id'].isin([831956, 1580627, 1580689, 1580626, 995177])]

# ‼️ 날자 필터링
accounts_blockrecord = accounts_blockrecord[accounts_blockrecord['created_at'] < '2023-09-01']

# head
accounts_blockrecord.head()

,user_id,block_user_id,reason,created_at
0,878476,867483,그냥...,2023-05-04 23:01:53
1,867564,867190,친구 사이가 어색해짐,2023-05-05 01:17:08
2,875261,875110,나랑 관련 없는 질문을 자꾸 보냄,2023-05-05 01:50:55
3,883511,883696,그냥...,2023-05-05 05:21:52
4,870177,871349,그냥...,2023-05-05 06:40:34


In [94]:
# 한달이내 누적 신고수가 상위 2퍼 넘을경우 유의 인물 추가
accounts_blockrecord['reason'].unique()

array(['그냥...', '친구 사이가 어색해짐', '나랑 관련 없는 질문을 자꾸 보냄', '기타', '모르는 사람임',
       '너무 많은 양의 질문을 보냄', '사칭 계정'], dtype=object)

In [92]:
cond1 = (accounts_blockrecord['created_at'] >= '2023-05-01')
cond2 = (accounts_blockrecord['created_at'] < '2023-06-01')
block_23_05 = (accounts_blockrecord[cond1&cond2]['block_user_id'].value_counts(normalize=True)*100).reset_index()
block_23_05['prop_cumsm'] = block_23_05['proportion'].cumsum()
df = block_23_05[block_23_05['prop_cumsm'] < 2]
df['percent'] = (df['proportion'] / df['proportion'].sum() )* 100
df

,block_user_id,proportion,prop_cumsm,percent
0,898020,0.462371,0.462371,23.899371
1,876207,0.146012,0.608384,7.547170
2,1395312,0.139928,0.748312,7.232704
3,877266,0.121677,0.869988,6.289308
4,897681,0.115593,0.985581,5.974843
5,1495281,0.109509,1.095090,5.660377
6,1031842,0.109509,1.204599,5.660377
7,1186363,0.103425,1.308025,5.345912
8,1198628,0.091258,1.399282,4.716981
9,957885,0.091258,1.490540,4.716981


In [ ]:
cond1 = (accounts_blockrecord['created_at'] >= '2023-06-01')
cond2 = (accounts_blockrecord['created_at'] < '2023-07-01')
block_23_06 = accounts_blockrecord[cond1&cond2]
(block_23_06['block_user_id'].value_counts(normalize=True)*100).reset_index()

,block_user_id,proportion
0,1575252,0.663130
1,1380465,0.530504
2,1533716,0.486295
3,1542237,0.353669
4,1571949,0.353669
...,...,...
2080,1336181,0.044209
2081,1402912,0.044209
2082,1126167,0.044209
2083,1564549,0.044209


In [74]:
cond1 = (accounts_blockrecord['created_at'] >= '2023-07-01')
cond2 = (accounts_blockrecord['created_at'] < '2023-08-01')
block_23_07 = accounts_blockrecord[cond1&cond2]
(block_23_07['block_user_id'].value_counts(normalize=True)*100).reset_index()

,block_user_id,proportion
0,1345082,0.731707
1,1571977,0.731707
2,1115754,0.487805
3,1147132,0.487805
4,971604,0.487805
...,...,...
383,900672,0.243902
384,1578220,0.243902
385,1003597,0.243902
386,1111918,0.243902


In [75]:
cond1 = (accounts_blockrecord['created_at'] >= '2023-08-01')
cond2 = (accounts_blockrecord['created_at'] < '2023-09-01')
block_23_08 = accounts_blockrecord[cond1&cond2]
(block_23_08['block_user_id'].value_counts(normalize=True)*100).reset_index()

,block_user_id,proportion
0,1088412,3.773585
1,992181,1.886792
2,908710,1.886792
3,1500131,1.257862
4,951564,1.257862
...,...,...
142,1574348,0.628931
143,1392973,0.628931
144,1370152,0.628931
145,1082731,0.628931


In [57]:
(accounts_blockrecord['block_user_id'].value_counts(normalize=True)*100).reset_index()

,block_user_id,proportion
0,898020,0.394457
1,877266,0.129756
2,897681,0.129756
3,1380465,0.129756
4,1395312,0.124565
...,...,...
16043,1366300,0.005190
16044,1185927,0.005190
16045,1259362,0.005190
16046,1255481,0.005190


In [ ]:
# 이유별 누적 신고수 카운트
blocked_count = accounts_blockrecord.groupby(['block_user_id', 'reason'])['user_id'].nunique().reset_index(name='blocked_count')\
    .sort_values(by=['block_user_id', 'blocked_count'], ascending=False)
blocked_count




,block_user_id,reason,blocked_count
1110,898020,모르는 사람임,13
1109,898020,너무 많은 양의 질문을 보냄,9
1108,898020,나랑 관련 없는 질문을 자꾸 보냄,6
1112,898020,친구 사이가 어색해짐,6
1111,898020,사칭 계정,4


In [ ]:
# 한달이나 누적 신고수가 5건이 넘을경우 유의 인물 추가

In [ ]:
# 하루동안 신고 누적 카운트

# 만약 하루안에 특정 한명이 누군가를 3번이상 신고했다면 유의인물 추가

In [48]:
accounts_blockrecord.query('block_user_id == 898020 & reason == "모르는 사람임"')['user_id'].nunique()

13

In [37]:
accounts_blockrecord['block_user_id'].value_counts().reset_index()

,block_user_id,count
0,898020,76
1,877266,25
2,897681,25
3,1380465,25
4,1395312,24
...,...,...
16043,1366300,1
16044,1185927,1
16045,1259362,1
16046,1255481,1
